In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import jv

# =========================
# Constants
# =========================
INF = np.inf
hbar = 1
e = 1
beta = INF  # Zero temperature case: T = 0, beta -> infinity
phi_quantum = 1 / 2

# =========================
# Characterization of the circuit
# =========================
omega_a = 25.338776456203686
omega_b = 2 * omega_a

phi_a = 0.11
Z_a = 2 * phi_quantum**2 * phi_a**2
C_a_inv = Z_a * omega_a
E_Ca = C_a_inv / 2

phi_b = 0.204
Z_b = 2 * phi_quantum**2 * phi_a**2   # kept exactly as in your Julia code
C_b_inv = Z_b * omega_b
E_Cb = C_b_inv / 2

E_J = 37.12 * 2 * np.pi * 1.5

# =========================
# Environment
# =========================
kappa_b = 1 / 10.4  # 1/ns from Leon's paper

omega_d = omega_b                 # Drive frequency
omega_p = 2 * omega_a - omega_d   # Pump frequency

epsilon_p = 0.2
g2 = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b  # Leon's paper
epsilon_d = 10.12 * g2  # 0.75 * g2_max

beta_a_p = 2 * E_J * np.sin(epsilon_p) * phi_a**2 / omega_a
beta_b_p = 2 * E_J * np.sin(epsilon_p) * phi_b**2 / omega_b
epsilon_d_p = epsilon_d * phi_b / omega_b
kappa_b_p = kappa_b / omega_b

aleph = beta_a_p * beta_b_p / (4 * kappa_b_p)
beth = beta_a_p * epsilon_d_p / (2 * kappa_b_p)

M = 200
eps = 1e-3

phi_left = -np.sqrt(2 * epsilon_d_p / beta_b_p)
phi_right = 0.0
phi_grid = np.linspace(phi_left, phi_right, M + 1)

alpha = 0.01
theta_best = (aleph * phi_grid**3 - beth * phi_grid) / eps   # = -b(phi)/eps
theta_init = np.zeros(M + 1)

u = phi_grid + alpha * theta_init
v = phi_grid - alpha * theta_init

delta_s = 1 / M
delta_tau = 1e-2
N_iter = int(1e3)

# =========================
# Main iteration loop
# =========================
for n in range(1, N_iter + 1):
    # ---------- pass 1: build lambda(u^n, v^n), update u ----------
    p = (u + v) / 2
    Pp = (u - v) / (2 * alpha)

    grad_theta = (-aleph * p**3 + beth * p) + 2 * eps * Pp
    grad_phi = (-3 * aleph * p**2 + beth) * Pp
    Hvals = ((-aleph * p**3 + beth * p) * Pp) + eps * Pp**2
    fvals = alpha * grad_phi - grad_theta

    ds_phi = np.zeros(M + 1)
    ds_phi[0] = (p[1] - p[0]) / delta_s
    for i in range(1, M):
        ds_phi[i] = (p[i + 1] - p[i - 1]) / (2 * delta_s)
    ds_phi[-1] = (p[-1] - p[-2]) / delta_s

    norm2 = ds_phi**2
    dot_term = grad_theta * ds_phi
    disc = dot_term**2 - 4 * Hvals * norm2
    lambda_vals = (np.abs(dot_term) + np.sqrt(np.maximum(0.0, disc))) / (
        2 * np.maximum(norm2, 1e-14)
    )

    lambda_max = np.max(lambda_vals)

    u_new = np.empty_like(u)
    u_new[-1] = -v[-1] + 2 * phi_right
    for i in range(M - 1, -1, -1):
        u_new[i] = u[i] + (
            lambda_vals[i] * (u_new[i + 1] - u[i]) / delta_s + fvals[i + 1]
        ) * delta_tau

    # ---------- pass 2: build lambda(u^{n+1}, v^n), update v ----------
    p = (u_new + v) / 2
    Pp = (u_new - v) / (2 * alpha)

    grad_theta = (-aleph * p**3 + beth * p) + 2 * eps * Pp
    grad_phi = (-3 * aleph * p**2 + beth) * Pp
    Hvals = ((-aleph * p**3 + beth * p) * Pp) + eps * Pp**2
    gvals = alpha * grad_phi + grad_theta

    ds_phi = np.zeros(M + 1)
    ds_phi[0] = (p[1] - p[0]) / delta_s
    for i in range(1, M):
        ds_phi[i] = (p[i + 1] - p[i - 1]) / (2 * delta_s)
    ds_phi[-1] = (p[-1] - p[-2]) / delta_s

    norm2 = ds_phi**2
    dot_term = grad_theta * ds_phi
    disc = dot_term**2 - 4 * Hvals * norm2
    lambda_vals = (np.abs(dot_term) + np.sqrt(np.maximum(0.0, disc))) / (
        2 * np.maximum(norm2, 1e-14)
    )

    v_new = np.empty_like(v)
    v_new[0] = -u_new[0] + 2 * phi_left
    for i in range(1, M + 1):
        lambda_i = lambda_vals[i]
        v_new[i] = v[i] - (
            lambda_i * (v[i] - v_new[i - 1]) / delta_s - gvals[i]
        ) * delta_tau

    # ---------- robust 1D reparam on fixed phi-grid ----------
    phi = (u_new + v_new) / 2
    theta = (u_new - v_new) / (2 * alpha)

    # sort by phi so interpolation is well-defined
    perm = np.argsort(phi)
    phi_s = phi[perm]
    theta_s = theta[perm]

    theta_r = np.empty_like(theta)

    # preserve endpoints exactly
    theta_r[0] = theta_s[0]
    theta_r[-1] = theta_s[-1]

    for j in range(1, M):
        phi_t = phi_grid[j]
        k = np.searchsorted(phi_s, phi_t, side="left")

        if k <= 0:
            theta_r[j] = theta_s[0]
        elif k > M:
            theta_r[j] = theta_s[-1]
        else:
            phi0, phi1 = phi_s[k - 1], phi_s[k]
            t = 0.0 if abs(phi1 - phi0) < 1e-14 else (phi_t - phi0) / (phi1 - phi0)
            theta_r[j] = (1 - t) * theta_s[k - 1] + t * theta_s[k]

    theta_r[0] = 0.0
    theta_r[-1] = 0.0

    u = phi_grid + alpha * theta_r
    v = phi_grid - alpha * theta_r

    if n % 200 == 1:
        pchk = (u + v) / 2
        Pchk = (u - v) / (2 * alpha)
        Hchk = ((-aleph * pchk**3 + beth * pchk) * Pchk) + eps * Pchk**2
        print(f"iter={n}, max|H|={np.max(np.abs(Hchk))}")

# =========================
# Final solution
# =========================
p = (u + v) / 2
P = (u - v) / (2 * alpha)

phi_all = np.arange(-0.25, 0.0100001, 0.001)
theta_all = np.arange(-0.1, 7.0000001, 0.1)

def H(p, P_p, params):
    aleph_, beth_ = params
    return P_p * (-aleph_ * (p**2) * p + beth_ * p) + eps * P_p**2

H_val = np.array([
    [H(phi_val, P_val, (aleph, beth)) for P_val in theta_all]
    for phi_val in phi_all
])

# =========================
# Plot
# =========================
Theta, Phi = np.meshgrid(theta_all, phi_all)

plt.figure(figsize=(8, 6))
plt.pcolormesh(Theta, Phi, H_val, shading="auto")
plt.colorbar(label="H")
plt.xlabel(r"$P_p$")
plt.ylabel(r"$p$")

contours = plt.contour(Theta, Phi, H_val, levels=[0.0], colors="white", linewidths=2)
plt.clabel(contours, inline=True, fontsize=8)

# p_stable was referenced in the Julia snippet but not defined there.
# Uncomment if you define it:
# plt.plot(np.zeros(len(p_stable)), p_stable, "r--", label="Noise-free dissipation")

plt.plot(theta_best, phi_grid, "--", label="Optimal path")
plt.scatter(P, p, s=5, label="Solution of the GDA", c='r')

plt.legend()
plt.tight_layout()
plt.show()